# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [10]:
## import asyncio
import json
import os
import time
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from openai import AsyncOpenAI

# Load environment variables from .env file
load_dotenv()

# Make sure your OPENAI_API_KEY is set in the environment
# assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'
print('Loading environment variables...')
if os.getenv("OPENROUTER_API_KEY"):
   API_KEY = os.getenv("OPENROUTER_API_KEY")
   URL_BASE = os.getenv("OPENROUTER_BASE")
else:
    assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'
    API_KEY = os.environ.get('OPENAI_API_KEY')
    URL_BASE = "https://openai.vocareum.com/v1"

client = AsyncOpenAI(
    base_url=URL_BASE,
    api_key=API_KEY
)

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Loading environment variables...
Setup complete.


## Step 1 — Load the data

In [11]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [12]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    return [{
        'role': 'user',
        'content': (
            'Extract the company, role, and years of experience required from the job snippet below. '
            'Return only a valid JSON object with these exact fields: '
            'company, role, years_experience_required. '
            'Use null when a value is not explicitly stated; do not infer or guess. '
            f'Job snippet: {snippet_text}'
        ),
    }]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    return [{
        'role': 'user',
        'content': (
            'Extract the company, role, and years of experience required from each job snippet. '
            'Return only a valid JSON object with these exact fields: '
            'company, role, years_experience_required. '
            'Use the stated number for phrases such as "7+ years" or "around 6 years". '
            'If a value is not explicitly stated, use null; do not infer or guess.\n\n'
            'Example 1:\n'
            'Job snippet: "Northstar Labs is hiring a Data Analyst. Candidates need 3 years of experience."\n'
            'Output: {"company": "Northstar Labs", "role": "Data Analyst", "years_experience_required": 3}\n\n'
            'Example 2:\n'
            'Job snippet: "Join BrightPath as a Product Designer. We do not list a specific years requirement."\n'
            'Output: {"company": "BrightPath", "role": "Product Designer", "years_experience_required": null}\n\n'
            f'Now extract from this job snippet:\n{snippet_text}'
        ),
    }]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    return [
        {
            'role': 'system',
            'content': (
                'You are an expert technical recruiter performing precise information extraction. '
                'Return only one valid JSON object and no markdown or explanation. '
                'The JSON must follow this schema: '
                '{"company": string, "role": string, "years_experience_required": integer or null}. '
                'Use the minimum stated number for ranges such as "between 3 and 5 years". '
                'Use the stated number for phrases such as "7+ years" or "around 6 years". '
                'If a field is not explicitly stated in the snippet, return null for that field. '
                'Never infer or guess missing information.'
            ),
        },
        {
            'role': 'user',
            'content': f'Extract the required fields from this job snippet:\n{snippet_text}',
        },
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [{
        'role': 'user',
        'content': (
            'Analyse the job snippet step by step. First identify the company and role, then identify '
            'the explicitly stated experience requirement. Use the minimum number for ranges, and use '
            'the stated number for phrases such as "7+ years" or "around 6 years". '
            'If the snippet does not explicitly state a value, use null rather than inferring or guessing. '
            'After your reasoning, end with one valid JSON object containing exactly these fields: '
            'company, role, years_experience_required.\n\n'
            f'Job snippet:\n{snippet_text}'
        ),
    }]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [13]:
import asyncio


def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.

    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    if not isinstance(text, str) or not text.strip():
        return None

    without_fences = '\n'.join(
        line for line in text.splitlines()
        if not line.strip().startswith('```')
    ).strip()

    try:
        parsed = json.loads(without_fences)
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        pass

    decoder = json.JSONDecoder()
    for index, character in enumerate(without_fences):
        if character != '{':
            continue
        try:
            parsed, _ = decoder.raw_decode(without_fences, index)
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            return parsed

    return None


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    messages = STRATEGIES[strategy_name](snippet['snippet'])
    started_at = time.perf_counter()

    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
    )

    latency_s = time.perf_counter() - started_at
    raw_response = response.choices[0].message.content or ''
    extracted = parse_response(raw_response)
    usage = response.usage
    input_tokens = int(getattr(usage, 'prompt_tokens', 0) or 0)
    output_tokens = int(getattr(usage, 'completion_tokens', 0) or 0)
    rate = RATES[MODEL]
    cost_usd = (input_tokens * rate['in']) + (output_tokens * rate['out'])

    return {
        'strategy': strategy_name,
        'snippet_id': snippet['id'],
        'raw_response': raw_response,
        'extracted': extracted,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'cost_usd': cost_usd,
        'latency_s': latency_s,
    }


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = [
        run_one(strategy_name, snippet)
        for strategy_name in STRATEGIES
        for snippet in snippets
    ]
    return list(await asyncio.gather(*tasks))

In [14]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '```json\n{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}\n```',
 'extracted': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'input_tokens': 99,
 'output_tokens': 34,
 'cost_usd': 3.5249999999999996e-05,
 'latency_s': 1.9735254000406712}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [15]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    if not isinstance(extracted, dict):
        return 0

    def normalise(value, field: str):
        if field == 'years_experience_required':
            if value is None:
                return None
            if isinstance(value, bool):
                return value
            if isinstance(value, int):
                return value
            if isinstance(value, float):
                return int(value) if value.is_integer() else value
            value = str(value).strip().rstrip('+').strip()
            try:
                return int(value)
            except ValueError:
                return value.casefold()
        if value is None:
            return None
        return str(value).strip().casefold()

    fields = ('company', 'role', 'years_experience_required')
    return sum(
        normalise(extracted.get(field), field) == normalise(gold.get(field), field)
        for field in fields
    )


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    judge_messages = [
        {
            'role': 'system',
            'content': (
                'You are a strict evaluator for structured job-posting extraction. '
                'Compare the model extraction with the gold answer using the rubric. '
                'Return only one integer: 1, 2, 3, or 4. Do not include explanation.'
            ),
        },
        {
            'role': 'user',
            'content': (
                'Rubric:\n'
                '4 = all three fields correct.\n'
                '3 = two fields correct and no fabricated data.\n'
                '2 = one field correct, or fabricated a field.\n'
                '1 = none correct or extraction is unparsable.\n\n'
                f'Job snippet:\n{snippet_text}\n\n'
                f'Gold answer:\n{json.dumps(gold, ensure_ascii=True)}\n\n'
                f'Model extraction:\n{json.dumps(extracted, ensure_ascii=True)}'
            ),
        },
    ]

    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=judge_messages,
        temperature=0.0,
    )
    judge_text = (response.choices[0].message.content or '').strip()

    for token in judge_text.replace('.', ' ').split():
        if token in {'1', '2', '3', '4'}:
            return int(token)

    accuracy = score_accuracy(extracted, gold)
    return min(4, accuracy + 1) if accuracy else 1

In [16]:
async def score_result(result: dict) -> dict:
    """Attach deterministic and LLM-based scores to one result row."""
    snippet = next(item for item in snippets if item['id'] == result['snippet_id'])
    gold = golden[result['snippet_id']]
    row = dict(result)
    row['accuracy'] = score_accuracy(result['extracted'], gold)
    row['parse_success'] = result['extracted'] is not None
    row['llm_judge_score'] = await score_llm_judge(
        snippet['snippet'],
        result['extracted'],
        gold,
    )
    return row


scored = list(await asyncio.gather(*(score_result(result) for result in results)))
print(f'Scored {len(scored)} results.')

Scored 40 results.


In [ ]:
scored[0]

{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '```json\n{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}\n```',
 'extracted': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'input_tokens': 99,
 'output_tokens': 34,
 'cost_usd': 3.5249999999999996e-05,
 'latency_s': 2.8768359000096098,
 'accuracy': 3,
 'parse_success': True,
 'llm_judge_score': 4}

## Step 5 — Build the comparison table

In [17]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(6)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.7,1.0,3.8,0.001054,5.967515
few_shot,3.0,1.0,4.0,0.000494,4.093032
structured,3.0,1.0,3.8,0.000382,4.389013
zero_shot,2.6,1.0,3.6,0.000362,3.489297


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```